# Mood-Based Music Recommender System - ML Training Notebook

This notebook explains the machine learning workflow used in the Mood-Based Music Recommender System. It is written in a beginner-friendly way for college viva and project demonstration.

## 1. Import Required Libraries

We import the libraries needed for data handling, visualization, preprocessing, and K-Means clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (9, 5)

## 2. Dataset Loading

In this step, we load the `songs.csv` dataset. The dataset contains song details such as song name, artist, mood, energy level, and genre.

In [ ]:
df = pd.read_csv("songs.csv")
df.head()

### Dataset Information

The `info()` function shows the number of rows, columns, data types, and non-null values in the dataset.

In [ ]:
df.info()

### Null Values Check

Before training a model, we check whether any column has missing values.

In [ ]:
df.isnull().sum()

## 3. Preprocessing

Machine learning algorithms work with numbers, so we convert categorical text columns into numerical values using `LabelEncoder`.

Here, we encode:
- mood
- genre
- energy level

In [ ]:
processed_df = df.copy()

mood_encoder = LabelEncoder()
genre_encoder = LabelEncoder()
energy_encoder = LabelEncoder()

processed_df["mood_encoded"] = mood_encoder.fit_transform(processed_df["mood"])
processed_df["genre_encoded"] = genre_encoder.fit_transform(processed_df["genre"])
processed_df["energy_encoded"] = energy_encoder.fit_transform(processed_df["energy"])

processed_df.head()

## 4. Feature Selection

For clustering, we select the encoded columns because K-Means needs numerical input. These features represent the song's mood, genre, and energy level.

In [ ]:
features = processed_df[["mood_encoded", "genre_encoded", "energy_encoded"]]
features.head()

## 5. K-Means Clustering

K-Means is an unsupervised learning algorithm. It groups similar data points into clusters without using a target label.

For this project, we create **4 clusters** and assign each song to one cluster.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
processed_df["cluster"] = kmeans.fit_predict(features)

processed_df[["song", "artist", "mood", "energy", "genre", "cluster"]].head()

## 6. Visualization - Elbow Method

The elbow method helps us understand how the K-Means inertia changes for different numbers of clusters. A lower inertia means songs are closer to their cluster centers.

In [ ]:
inertia_values = []
cluster_range = range(1, 11)

for k in cluster_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(features)
    inertia_values.append(model.inertia_)

plt.plot(cluster_range, inertia_values, marker="o", color="#8b5cf6")
plt.title("Elbow Method for Choosing Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.xticks(cluster_range)
plt.show()

## 7. Visualization - Scatter Plot of Clusters

This scatter plot shows how songs are grouped into clusters. Each color represents a different cluster created by K-Means.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=processed_df,
    x="mood_encoded",
    y="genre_encoded",
    hue="cluster",
    size="energy_encoded",
    palette="viridis",
    sizes=(80, 240),
)

for index, row in processed_df.iterrows():
    plt.text(
        row["mood_encoded"] + 0.03,
        row["genre_encoded"] + 0.03,
        row["song"],
        fontsize=8,
        alpha=0.75,
    )

plt.title("K-Means Clusters of Songs")
plt.xlabel("Mood Encoded")
plt.ylabel("Genre Encoded")
plt.legend(title="Cluster")
plt.show()

## 8. Cluster Summary

This summary helps us understand how many songs are present in each cluster.

In [ ]:
processed_df["cluster"].value_counts().sort_index()

## 9. Conclusion

This is an **unsupervised learning project** because the model does not use a predefined target output. Instead, K-Means finds patterns in the song dataset by grouping similar songs together.

K-Means groups songs into clusters based on encoded mood, genre, and energy level. Songs inside the same cluster are considered more similar to each other.

In the Mood-Based Music Recommender System, recommendations are generated from cluster similarity. When a user selects a mood, genre, and energy level, the app finds the closest cluster and recommends songs from that similar group.